# 3 Baseline Multi-Label Classifier

As a baseline, we trained a simple multi-label text classifier using TF-IDF features and silver labels generated from keyword matching. The model is a feed-forward neural network with sigmoid outputs and binary cross-entropy loss, optimized using Adam and early stopping based on validation micro-F1. This baseline allows us to assess whether the automatically generated silver labels contain meaningful signal and establishes a reference point for future improvements such as pseudo-labeling or hierarchy-aware training.

In [9]:
# Relevant imports
import os
import csv
import copy
import random
from tqdm import tqdm
from collections import defaultdict
from pathlib import Path
import numpy as np
import pickle

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split

from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

# Random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Setup initial filepaths
ROOT = Path("project_release")

TRAIN_CORPUS_PATH = ROOT / "Amazon_products" / "train" / "train_corpus.txt"
TEST_CORPUS_PATH = ROOT / "Amazon_products" / "test" / "test_corpus.txt"
SILVER_PATH = ROOT / "silver_labels"

SUBMISSION_PATH = ROOT / "submissions"
SUBMISSION_PATH.mkdir(exist_ok=True)



In [10]:
# -----------------------------
# Load silver labels (choose one)
# -----------------------------

# TF-IDF silver labels
with open(SILVER_PATH / "silver_labels_tfidf.pkl", "rb") as f:
    silver_labels = pickle.load(f)

# SVD silver labels
#with open(SILVER_PATH / "silver_labels_svd.pkl", "rb") as f:
#    silver_labels = pickle.load(f)

# BERT silver labels
#with open(SILVER_PATH / "silver_labels_bert.pkl", "rb") as f:
#    silver_labels = pickle.load(f)
    
# Hybrid silver labels
# with open(SILVER_PATH / "silver_labels_hybrid.pkl", "rb") as f:
#     silver_labels = pickle.load(f)


In [11]:
# -----------------------------
# Load TF-IDF vectors for train corpus
# -----------------------------
TFIDF_PATH = ROOT / "representations" / "tfidf"
with open(TFIDF_PATH / "train_tfidf.pkl", "rb") as f:
    train_tfidf = pickle.load(f)  # scipy sparse matrix

train_tfidf = torch.tensor(train_tfidf.toarray(), dtype=torch.float32)  # convert to dense tensor

# -----------------------------
# Convert TF-IDF silver labels to dict
# -----------------------------
# TF-IDF silver labels are already {pid: [class_name, ...]}
review_to_classes = silver_labels  # no need to convert

# -----------------------------
# Dataset using TF-IDF features and silver labels
# -----------------------------
class ReviewsDataset(Dataset):
    def __init__(self, review_to_classes, features, class_to_idx):
        """
        review_to_classes: {pid: [class_name, ...]}
        features: torch tensor of shape (num_samples, input_dim)
        class_to_idx: {class_name: index}
        """
        self.review_ids = list(review_to_classes.keys())
        self.features = features
        self.class_to_idx = class_to_idx
        
        self.labels = []
        for pid in self.review_ids:
            label = [0] * len(class_to_idx)
            for cls in review_to_classes[pid]:
                if cls in class_to_idx:
                    label[class_to_idx[cls]] = 1
            self.labels.append(label)
        
        self.labels = torch.tensor(self.labels, dtype=torch.float32)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {"X": self.features[idx], "y": self.labels[idx]}

# Build class_to_idx mapping from silver labels
all_classes = sorted({cls for classes in review_to_classes.values() for cls in classes})
class_to_idx = {cls: idx for idx, cls in enumerate(all_classes)}

# Instantiate dataset
dataset = ReviewsDataset(review_to_classes, train_tfidf, class_to_idx)

# -----------------------------
# Train / validation split
# -----------------------------
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# -----------------------------
# Multi-label classifier
# -----------------------------
input_dim = train_tfidf.shape[1]
output_dim = len(class_to_idx)

model = MultiLabelClassifier(input_dim, output_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

print(f"Dataset ready: {len(dataset)} samples, {input_dim} features, {output_dim} classes")


AttributeError: 'list' object has no attribute 'values'

In [8]:
# -----------------------------
# Full Multi-label Training with TF-IDF Silver Labels
# -----------------------------

import copy
from sklearn.metrics import f1_score
import torch
import torch.nn.functional as F

# -----------------------------
# Hyperparameters
# -----------------------------
epochs = 20
patience = 5
batch_size = 32
threshold = 0.5
lr = 1e-3

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Load TF-IDF features
# -----------------------------
TFIDF_PATH = ROOT / "representations" / "tfidf"
with open(TFIDF_PATH / "train_tfidf.pkl", "rb") as f:
    train_tfidf = pickle.load(f)  # sparse matrix

train_tfidf = torch.tensor(train_tfidf.toarray(), dtype=torch.float32)

# -----------------------------
# Silver labels already loaded
# review_to_classes = silver_labels
# -----------------------------
all_classes = sorted({cls for classes in review_to_classes.values() for cls in classes})
class_to_idx = {cls: idx for idx, cls in enumerate(all_classes)}

# -----------------------------
# Dataset
# -----------------------------
class ReviewsDataset(torch.utils.data.Dataset):
    def __init__(self, review_to_classes, features, class_to_idx):
        self.review_ids = list(review_to_classes.keys())
        self.features = features
        self.class_to_idx = class_to_idx
        
        self.labels = []
        for pid in self.review_ids:
            label = [0] * len(class_to_idx)
            for cls in review_to_classes[pid]:
                if cls in class_to_idx:
                    label[class_to_idx[cls]] = 1
            self.labels.append(label)
        
        self.labels = torch.tensor(self.labels, dtype=torch.float32)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {"X": self.features[idx], "y": self.labels[idx]}

dataset = ReviewsDataset(review_to_classes, train_tfidf, class_to_idx)

# Train / Validation split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size)

# -----------------------------
# Model
# -----------------------------
class MultiLabelClassifier(torch.nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = torch.nn.Linear(input_dim, 512)
        self.fc2 = torch.nn.Linear(512, 256)
        self.out = torch.nn.Linear(256, output_dim)
        self.dropout = torch.nn.Dropout(0.3)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.out(x)
        return torch.sigmoid(x)

input_dim = train_tfidf.shape[1]
output_dim = len(class_to_idx)

model = MultiLabelClassifier(input_dim, output_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = torch.nn.BCELoss()

# -----------------------------
# Training loop with early stopping
# -----------------------------
best_val_f1 = -1
patience_counter = 0
best_model_state = None

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        X, y = batch["X"].to(device), batch["y"].to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"[Epoch {epoch}] Train Loss: {avg_loss:.4f}")

    # ---------- Validation ----------
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            X, y = batch["X"].to(device), batch["y"].to(device)
            logits = model(X)
            preds = (logits > threshold).int()
            all_preds.append(preds.cpu())
            all_labels.append(y.cpu().int())

    all_preds = torch.cat(all_preds, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy()

    val_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    print(f"[VAL] f1_micro: {val_f1:.4f}")

    # Early stopping
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("[Early Stopping] No improvement.")
        break

# Save best model
torch.save(best_model_state, "best_multilabel_model.pth")
print("[DONE] Training complete. Best f1_micro:", best_val_f1)




KAGGLE SUBMISSION

In [5]:
# ---------- Prepare dataset ----------
class_names = sorted({cls for classes in review_to_classes.values() for cls in classes})
class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

labels = []
review_ids = list(review_to_classes.keys())
for pid in review_ids:
    label = [0] * len(class_names)
    for cls in review_to_classes[pid]:
        if cls in class_to_idx:
            label[class_to_idx[cls]] = 1
    labels.append(label)

labels = torch.tensor(labels, dtype=torch.float32)
features = torch.tensor(train_dense, dtype=torch.float32)  # your SVD or TF-IDF embeddings

class ReviewsDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {"X": self.features[idx], "y": self.labels[idx]}

dataset = ReviewsDataset(features, labels)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)

# ---------- Model ----------
input_dim = features.shape[1]
output_dim = len(class_names)

class MultiLabelClassifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, 256)
        self.out = nn.Linear(256, output_dim)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        return self.out(x)  # logits

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MultiLabelClassifier(input_dim, output_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()  # logits + BCE

# ---------- Training loop ----------
EPOCHS = 10
patience = 5
best_val_f1 = -1
patience_counter = 0
best_model_state = None

for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        X, y = batch["X"].to(device), batch["y"].to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"[Epoch {epoch}] Train Loss: {avg_loss:.4f}")

    # ---------- Validation ----------
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            X, y = batch["X"].to(device), batch["y"].to(device)
            logits = model(X)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()
            all_preds.append(preds.cpu())
            all_labels.append(y.cpu().int())

    all_preds = torch.cat(all_preds, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy()
    val_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    print(f"[VAL] f1_micro: {val_f1:.4f}")

    # ---------- Early stopping ----------
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("[Early Stopping] No improvement.")
        break

# Save best model
torch.save(best_model_state, SILVER_PATH / "last_best.pth")
print("[DONE] Training complete. Best f1_micro:", best_val_f1)


NameError: name 'train_dense' is not defined